# Notebook 04 — Rescue-Aware Decision Engine for Predicted Late-Stage Clone Selection

## Goal

Convert predicted late-stage clone performance from Notebook03 into practical CLD selection decisions.

This notebook implements:

- pass / rescue / fail bucket assignment
- biosimilar and novel/ADC decision modes
- Stage 2 re-ranking
- final Top-10 / Top-3 clone selection
- rescue-aware decision validation

A rescue clone is not a final winner yet.  
It is a clone with enough upside to justify further process optimization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

scenario = "legacy"   # or "optimized"
n_clones = 5000

TOP_N = 10
TOP_STAGE2 = 3

print("Scenario:", scenario)
print("n_clones:", n_clones)
print("Top-N final:", TOP_N)
print("Top-N stage2:", TOP_STAGE2)

Scenario: legacy
n_clones: 5000
Top-N final: 10
Top-N stage2: 3


## Step 1 — Load prediction table from Notebook 03

Here we load the prediction table produced in Notebook 03b.

This table should already contain:

- predicted late qP
- predicted qP drop
- predicted late aggregation
- predicted stable probability
- true late outcomes for offline simulation only

In [2]:
PRED_PATH = Path("../data/synthetic/processed") / f"predictions_03b_qp_{n_clones}_{scenario}.csv"
print("Loading predictions:", PRED_PATH)

df = pd.read_csv(PRED_PATH)
print("Shape:", df.shape)
display(df.head())

Loading predictions: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Shape: (1000, 20)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,true_is_aggressive,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.381373,2.711177e-08,9.583127,0.370940,0,0.111023,4.176362e-08,12.566994,1,0.000000,0.024314,0,0.000518,0.895422,0.803068,0.974885,0.486344,0.598822,1
1,CLONE_2587,0.489493,6.445184e-08,3.127400,0.169707,0,0.691891,3.691256e-08,0.918511,0,0.000000,0.001179,0,0.007290,0.535023,0.000000,0.998782,0.769478,0.264869,0
2,CLONE_2654,0.362087,3.071979e-08,7.313653,0.492969,0,0.009193,4.002863e-08,6.558486,1,0.004355,0.014202,0,0.001172,0.959708,0.661044,0.985331,0.314649,0.571715,0
3,CLONE_1056,0.353752,6.793292e-08,6.868662,0.478848,0,0.708930,3.026369e-08,7.990409,0,0.004891,0.001310,0,0.007922,0.987492,0.533903,0.998647,0.334518,0.549744,0
4,CLONE_0706,0.616517,1.729449e-07,8.278383,0.036653,0,0.740570,1.029717e-07,7.441547,0,0.012486,0.072412,0,0.026969,0.111609,0.936681,0.925203,0.956685,0.397218,0


## Step 2 — Check schema compatibility

Before applying selection rules, we verify that the prediction table matches the expected 03b output schema.

This prevents silent notebook mismatch caused by:
- outdated file names
- old column naming
- partial notebook updates

In [3]:
required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_stable_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)
assert len(missing) == 0, f"04b is missing required columns: {missing}"

optional_cols = [
    "pred_super_prob",
    "pred_aggr_prob",
    "true_is_aggressive",
    "pred_rescue_score",
    "pred_rescue_label",
]

present_optional = [c for c in optional_cols if c in df.columns]
print("Optional columns found:", present_optional)

if "pred_rescue_score" not in df.columns:
    print("[WARN] pred_rescue_score missing. Setting to 0.0.")
    df["pred_rescue_score"] = 0.0

if "pred_rescue_label" not in df.columns:
    print("[WARN] pred_rescue_label missing. Setting to 0.")
    df["pred_rescue_label"] = 0

if "pred_aggr_prob" not in df.columns:
    print("[WARN] pred_aggr_prob missing. Setting to 0.0.")
    df["pred_aggr_prob"] = 0.0

if "true_is_aggressive" not in df.columns:
    print("[WARN] true_is_aggressive missing. Setting to 0 for offline FP evaluation.")
    df["true_is_aggressive"] = 0

df["true_is_aggressive"] = df["true_is_aggressive"].fillna(0).astype(int)

Missing required columns: []
Optional columns found: ['pred_super_prob', 'pred_aggr_prob', 'true_is_aggressive', 'pred_rescue_score', 'pred_rescue_label']


## Step 3 — Define helper functions

We use helper functions for:

- z-score normalization
- ranking utility calculation
- top-k overlap
- elite capture metrics

These help compare decision quality across selection modes.

In [4]:
def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def make_pred_utility(df, w_qp, w_drop, w_agg):
    return (
        w_qp * z(df["pred_late_qp"])
        - w_drop * z(df["pred_qp_drop"])
        - w_agg * z(df["pred_late_agg"])
    )

def make_true_utility(df, w_qp, w_drop, w_agg):
    return (
        w_qp * z(df["true_late_qp"])
        - w_drop * z(df["true_qp_drop"])
        - w_agg * z(df["true_late_agg"])
    )

def topk_overlap(true_scores, pred_scores, k):
    true_top = set(pd.Series(true_scores).nlargest(k).index)
    pred_top = set(pd.Series(pred_scores).nlargest(k).index)
    return len(true_top & pred_top) / k

def precision_at_k_df(df, score_col, elite_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    return float(top[elite_col].mean())

def ndcg_at_k_df(df, score_col, elite_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    rel = top[elite_col].values.astype(float)
    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))
    dcg = float(np.sum(rel * discounts))
    ideal = np.sort(rel)[::-1]
    idcg = float(np.sum(ideal * discounts)) + 1e-12
    return dcg / idcg

In [5]:
def summarize_bucket(df, bucket_col):
    return df.groupby(bucket_col)[[
        "pred_late_qp",
        "pred_qp_drop",
        "pred_late_agg",
        "pred_stable_prob",
        "pred_rescue_score",
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_stable_label",
    ]].mean()

def summarize_final_selection(name, top10, top3, bucket_col):
    return {
        "mode": name,
        "top10_n": len(top10),
        "top3_n": len(top3),
        "top10_rescue_count": int((top10[bucket_col] == "rescue").sum()),
        "top3_rescue_count": int((top3[bucket_col] == "rescue").sum()),
        "top10_true_stable_rate": float(top10["true_stable_label"].mean()),
        "top3_true_stable_rate": float(top3["true_stable_label"].mean()),
        "top10_mean_true_late_qp": float(top10["true_late_qp"].mean()),
        "top10_mean_true_qp_drop": float(top10["true_qp_drop"].mean()),
        "top10_mean_true_late_agg": float(top10["true_late_agg"].mean()),
        "top10_mean_pred_rescue_score": float(top10["pred_rescue_score"].mean()),
    }

def apply_rescue_guardrail(
    stage2_df,
    bucket_col,
    score_col,
    top_n=10,
    top_stage2=3,
    min_rescue_top10=1,
):
    """
    Keep Top3 as pure ranking.
    For Top10, use pure ranking by default.
    If no rescue clone appears in Top10, replace the lowest-ranked non-Top3 pass clone
    with the best rescue candidate.
    """
    ranked = stage2_df.sort_values(score_col, ascending=False).copy()

    # Top3 is always pure ranking
    top3 = ranked.head(top_stage2).copy()

    # Top10 starts as pure ranking
    top10 = ranked.head(top_n).copy()

    if min_rescue_top10 <= 0:
        return top10.reset_index(drop=True), top3.reset_index(drop=True)

    current_rescue_n = int((top10[bucket_col] == "rescue").sum())

    if current_rescue_n >= min_rescue_top10:
        return top10.reset_index(drop=True), top3.reset_index(drop=True)

    protected_ids = set(top3["clone_id"])
    top10_ids = set(top10["clone_id"])

    rescue_pool = stage2_df[
        (stage2_df[bucket_col] == "rescue") &
        (~stage2_df["clone_id"].isin(top10_ids)) &
        (~stage2_df["clone_id"].isin(protected_ids))
    ].copy()

    if len(rescue_pool) == 0:
        return top10.reset_index(drop=True), top3.reset_index(drop=True)

    need_n = min_rescue_top10 - current_rescue_n

    rescue_add = rescue_pool.sort_values(
        ["pred_rescue_score", score_col],
        ascending=[False, False]
    ).head(need_n)

    replace_candidates = top10[
        (~top10["clone_id"].isin(protected_ids)) &
        (top10[bucket_col] != "rescue")
    ].sort_values(score_col, ascending=True).head(len(rescue_add))

    top10_new = pd.concat(
        [
            top10[~top10["clone_id"].isin(replace_candidates["clone_id"])],
            rescue_add,
        ],
        ignore_index=True,
    ).sort_values(score_col, ascending=False).head(top_n)

    return top10_new.reset_index(drop=True), top3.reset_index(drop=True)

## Step 4 — Define offline evaluation utilities

Because this is a synthetic dataset, we can define an offline “true utility” score
to evaluate how good the selected clones really are.

This is **evaluation only** and would not be available in real deployment.

In [6]:
# biosimilar-style evaluation utility
W_BIO = dict(w_qp=1.0, w_drop=1.0, w_agg=0.2)

# novel / ADC-style evaluation utility
W_NOVEL = dict(w_qp=1.0, w_drop=1.0, w_agg=1.0)

df["true_util_bio"] = make_true_utility(df, **W_BIO)
df["true_util_novel"] = make_true_utility(df, **W_NOVEL)

df["pred_util_bio"] = make_pred_utility(df, **W_BIO)
df["pred_util_novel"] = make_pred_utility(df, **W_NOVEL)

display(df[[
    "clone_id",
    "true_util_bio", "pred_util_bio",
    "true_util_novel", "pred_util_novel"
]].head())

,clone_id,true_util_bio,pred_util_bio,true_util_novel,pred_util_novel
0,CLONE_1502,1.032173,-0.084743,-0.492968,-1.270080
1,CLONE_2587,-1.015188,-0.469116,0.279491,0.589272
2,CLONE_2654,1.877311,0.304273,1.806686,-0.092296
3,CLONE_1056,-1.526406,0.506410,-1.943665,0.264501
4,CLONE_0706,-1.615101,-1.881235,-1.899494,-2.613101


## Step 5A — Decision mode: Biosimilar

In biosimilar mode, the selection logic is:

1. remove clones with poor predicted quality
2. remove clones with low predicted stability probability
3. rank the remaining clones by predicted late qP

This reflects a setting where:
- productivity matters strongly
- stability matters strongly
- aggregation still matters, but slightly less than in novel mode

In [7]:
# ==================================================
# Biosimilar mode — rescue-aware 3-bucket engine
# ==================================================

BIO_AGG_PASS_MAX = 15.0
BIO_STABLE_PASS_MIN = 0.50

BIO_AGG_RESCUE_MAX = 18.0
BIO_STABLE_RESCUE_MIN = 0.25
BIO_RESCUE_SCORE_MIN = df["pred_rescue_score"].quantile(0.70)

BIO_STAGE2_PASS_N = 25
BIO_STAGE2_RESCUE_N = 5

bio = df.copy()

bio["bio_pass"] = (
    (bio["pred_late_agg"] <= BIO_AGG_PASS_MAX) &
    (bio["pred_stable_prob"] >= BIO_STABLE_PASS_MIN)
)

bio["bio_rescue"] = (
    (~bio["bio_pass"]) &
    (bio["pred_rescue_score"] >= BIO_RESCUE_SCORE_MIN) &
    (bio["pred_late_agg"] <= BIO_AGG_RESCUE_MAX) &
    (bio["pred_stable_prob"] >= BIO_STABLE_RESCUE_MIN)
)

bio["bio_bucket"] = "fail"
bio.loc[bio["bio_rescue"], "bio_bucket"] = "rescue"
bio.loc[bio["bio_pass"], "bio_bucket"] = "pass"

bio_stage1_pass = bio[bio["bio_bucket"] == "pass"].copy()
bio_stage1_rescue = bio[bio["bio_bucket"] == "rescue"].copy()
bio_stage1_fail = bio[bio["bio_bucket"] == "fail"].copy()

print("=== Biosimilar Stage 1 ===")
print(bio["bio_bucket"].value_counts())

print("\n=== Biosimilar bucket diagnostics ===")
display(summarize_bucket(bio, "bio_bucket"))

bio_stage2_pass = bio_stage1_pass.sort_values(
    ["pred_late_qp", "pred_stable_prob"],
    ascending=[False, False]
).head(BIO_STAGE2_PASS_N).copy()

bio_stage2_rescue = bio_stage1_rescue.sort_values(
    ["pred_rescue_score", "pred_late_qp"],
    ascending=[False, False]
).head(BIO_STAGE2_RESCUE_N).copy()

bio_stage2 = pd.concat([bio_stage2_pass, bio_stage2_rescue], ignore_index=True)

bio_stage2["bio_final_score"] = (
    z(bio_stage2["pred_late_qp"])
    - 0.5 * z(bio_stage2["pred_qp_drop"])
    - 0.2 * z(bio_stage2["pred_late_agg"])
    + 0.7 * z(bio_stage2["pred_rescue_score"])
)

BIO_MIN_RESCUE_TOP10 = 1

bio_top10, bio_top3 = apply_rescue_guardrail(
    stage2_df=bio_stage2,
    bucket_col="bio_bucket",
    score_col="bio_final_score",
    top_n=TOP_N,
    top_stage2=TOP_STAGE2,
    min_rescue_top10=BIO_MIN_RESCUE_TOP10,
)

print("\n=== Biosimilar Stage 2 / Final ===")
print("Stage2 pass:", len(bio_stage2_pass))
print("Stage2 rescue:", len(bio_stage2_rescue))
print("Final top10:", len(bio_top10))
print("Final top3:", len(bio_top3))

display(bio_top10[[
    "clone_id",
    "bio_bucket",
    "bio_final_score",
    "pred_rescue_score",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in bio_top10.columns]])

=== Biosimilar Stage 1 ===
bio_bucket
fail      636
rescue    225
pass      139
Name: count, dtype: int64

=== Biosimilar bucket diagnostics ===


,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_stable_label
bio_bucket,,,,,,,,,
fail,1.801443e-07,0.475321,5.499093,0.258937,0.346000,3.111778e-07,0.473110,5.555970,0.188679
pass,1.725751e-07,0.335535,5.564425,0.546728,0.427139,2.240865e-07,0.292263,5.939623,0.539568
rescue,1.333574e-07,0.392051,8.452233,0.391103,0.561926,1.529669e-07,0.382127,8.477936,0.333333



=== Biosimilar Stage 2 / Final ===
Stage2 pass: 25
Stage2 rescue: 5
Final top10: 10
Final top3: 3


,clone_id,bio_bucket,bio_final_score,pred_rescue_score,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_4625,rescue,5.292870,1.000000,5.537450e-06,0.357910,8.834887,0.486118,1.017815e-05,0.440572,11.935039,0,0.836577,0.028849
1,CLONE_3254,pass,4.383504,0.815263,4.310815e-06,0.310529,8.071338,0.621432,8.830813e-06,0.000000,9.991042,1,0.917478,0.008511
2,CLONE_1619,pass,2.002790,0.294760,2.226262e-06,0.252291,3.094684,0.677180,2.551172e-06,0.107071,0.670928,1,0.874076,0.001411
3,CLONE_2759,pass,0.178286,0.448798,8.089825e-07,0.283589,10.860110,0.591933,1.384358e-06,0.000000,12.804618,1,0.605596,0.004202
4,CLONE_3822,pass,-0.033496,0.149395,3.169120e-07,0.242531,3.335838,0.716600,4.217242e-07,0.084796,6.275180,1,0.447109,0.003124
5,CLONE_1834,pass,-0.212887,0.470103,3.203133e-07,0.310774,6.870380,0.599999,3.549060e-07,0.309623,8.811057,0,0.416910,0.000000
6,CLONE_2520,rescue,-0.263110,0.676299,1.574354e-07,0.346800,8.358086,0.460809,1.808132e-07,0.261930,7.855895,1,0.039147,0.000000
7,CLONE_1757,pass,-0.269827,0.282854,3.768247e-07,0.293559,3.116737,0.567000,4.434637e-07,0.242789,2.088970,1,0.156231,0.001427
8,CLONE_4616,pass,-0.282742,0.592751,2.329367e-07,0.335650,7.664888,0.530770,1.851350e-07,0.460358,10.842138,0,0.031166,0.000000
9,CLONE_2358,pass,-0.306563,0.404821,4.675453e-07,0.314394,5.801222,0.537958,4.478037e-07,0.403951,8.789003,0,0.234306,0.000000


## Step 6A — Novel / ADC 3-bucket decision engine

In novel / ADC mode, quality is emphasized more strongly than in biosimilar mode.

So compared with biosimilar mode:

- aggregation threshold is stricter
- stability threshold is slightly stricter
- rescue margin is narrower
- productivity still matters, but low-quality clones are filtered more aggressively

We use the same 3-bucket logic:

- **pass** = strong candidates that clearly meet the criteria
- **rescue** = borderline but potentially valuable clones
- **fail** = clones that should be dropped

In [8]:
# ==================================================
# Novel / ADC mode — rescue-aware 3-bucket engine
# ==================================================

NOVEL_AGG_PASS_MAX = 10.0
NOVEL_STABLE_PASS_MIN = 0.55

NOVEL_AGG_RESCUE_MAX = 13.0
NOVEL_STABLE_RESCUE_MIN = 0.30
NOVEL_RESCUE_SCORE_MIN = df["pred_rescue_score"].quantile(0.75)

NOVEL_STAGE2_PASS_N = 25
NOVEL_STAGE2_RESCUE_N = 5

novel = df.copy()

novel["novel_pass"] = (
    (novel["pred_late_agg"] <= NOVEL_AGG_PASS_MAX) &
    (novel["pred_stable_prob"] >= NOVEL_STABLE_PASS_MIN)
)

novel["novel_rescue"] = (
    (~novel["novel_pass"]) &
    (novel["pred_rescue_score"] >= NOVEL_RESCUE_SCORE_MIN) &
    (novel["pred_late_agg"] <= NOVEL_AGG_RESCUE_MAX) &
    (novel["pred_stable_prob"] >= NOVEL_STABLE_RESCUE_MIN)
)

novel["novel_bucket"] = "fail"
novel.loc[novel["novel_rescue"], "novel_bucket"] = "rescue"
novel.loc[novel["novel_pass"], "novel_bucket"] = "pass"

novel_stage1_pass = novel[novel["novel_bucket"] == "pass"].copy()
novel_stage1_rescue = novel[novel["novel_bucket"] == "rescue"].copy()
novel_stage1_fail = novel[novel["novel_bucket"] == "fail"].copy()

print("=== Novel / ADC Stage 1 ===")
print(novel["novel_bucket"].value_counts())

print("\n=== Novel / ADC bucket diagnostics ===")
display(summarize_bucket(novel, "novel_bucket"))

novel_stage2_pass = novel_stage1_pass.sort_values(
    ["pred_late_agg", "pred_late_qp", "pred_stable_prob"],
    ascending=[True, False, False]
).head(NOVEL_STAGE2_PASS_N).copy()

novel_stage2_rescue = novel_stage1_rescue.sort_values(
    ["pred_rescue_score", "pred_late_agg", "pred_late_qp"],
    ascending=[False, True, False]
).head(NOVEL_STAGE2_RESCUE_N).copy()

novel_stage2 = pd.concat([novel_stage2_pass, novel_stage2_rescue], ignore_index=True)

novel_stage2["novel_final_score"] = (
    z(novel_stage2["pred_late_qp"])
    - 0.6 * z(novel_stage2["pred_qp_drop"])
    - 1.0 * z(novel_stage2["pred_late_agg"])
    + 0.5 * z(novel_stage2["pred_rescue_score"])
)

NOVEL_MIN_RESCUE_TOP10 = 0

novel_top10, novel_top3 = apply_rescue_guardrail(
    stage2_df=novel_stage2,
    bucket_col="novel_bucket",
    score_col="novel_final_score",
    top_n=TOP_N,
    top_stage2=TOP_STAGE2,
    min_rescue_top10=NOVEL_MIN_RESCUE_TOP10,
)

print("\n=== Novel / ADC Stage 2 / Final ===")
print("Stage2 pass:", len(novel_stage2_pass))
print("Stage2 rescue:", len(novel_stage2_rescue))
print("Final top10:", len(novel_top10))
print("Final top3:", len(novel_top3))

display(novel_top10[[
    "clone_id",
    "novel_bucket",
    "novel_final_score",
    "pred_rescue_score",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_top10.columns]])

=== Novel / ADC Stage 1 ===
novel_bucket
fail      767
rescue    189
pass       44
Name: count, dtype: int64

=== Novel / ADC bucket diagnostics ===


,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_stable_label
novel_bucket,,,,,,,,,
fail,1.705511e-07,0.457964,5.615191,0.289097,0.363134,2.805935e-07,0.455865,5.676395,0.216428
pass,2.977614e-07,0.316608,5.709993,0.591902,0.412753,4.228300e-07,0.245388,6.029685,0.636364
rescue,1.304281e-07,0.380774,8.542535,0.428024,0.577658,1.569046e-07,0.354791,8.717669,0.402116



=== Novel / ADC Stage 2 / Final ===
Stage2 pass: 25
Stage2 rescue: 5
Final top10: 10
Final top3: 3


,clone_id,novel_bucket,novel_final_score,pred_rescue_score,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_4625,rescue,3.952541,1.000000,5.537450e-06,0.357910,8.834887,0.486118,1.017815e-05,0.440572,11.935039,0,0.836577,0.028849
1,CLONE_1619,pass,3.593900,0.294760,2.226262e-06,0.252291,3.094684,0.677180,2.551172e-06,0.107071,0.670928,1,0.874076,0.001411
2,CLONE_3822,pass,1.373811,0.149395,3.169120e-07,0.242531,3.335838,0.716600,4.217242e-07,0.084796,6.275180,1,0.447109,0.003124
3,CLONE_3627,pass,1.036008,0.253686,2.592586e-08,0.290414,2.320564,0.556708,2.805876e-08,0.000000,0.423692,1,0.012414,0.008159
4,CLONE_1757,pass,0.994533,0.282854,3.768247e-07,0.293559,3.116737,0.567000,4.434637e-07,0.242789,2.088970,1,0.156231,0.001427
5,CLONE_0231,pass,0.866416,0.271140,1.968749e-07,0.295512,2.890700,0.630171,1.941276e-07,0.308893,2.457631,0,0.022105,0.000000
6,CLONE_4292,pass,0.792511,0.290113,9.937740e-08,0.305242,2.609595,0.584531,5.861043e-08,0.610870,7.043120,0,0.026768,0.008436
7,CLONE_4484,pass,0.520769,0.294394,3.940902e-07,0.298721,3.952418,0.584305,5.302242e-07,0.158500,4.855400,1,0.265731,0.001553
8,CLONE_3249,pass,0.442046,0.338768,1.911850e-07,0.323057,3.103373,0.573172,2.377898e-07,0.138785,1.980966,1,0.012063,0.000000
9,CLONE_0080,pass,0.407464,0.319065,4.130405e-08,0.318777,2.929115,0.564021,5.556941e-08,0.071082,2.631481,1,0.001464,0.010233


## Step 7 — Compare decision modes

This step makes it easier to see how the selected clone sets differ
between biosimilar mode and novel / ADC mode.

In [9]:
# ==================================================
# Rescue-aware decision validation
# ==================================================

decision_validation = pd.DataFrame([
    summarize_final_selection("biosimilar", bio_top10, bio_top3, "bio_bucket"),
    summarize_final_selection("novel/ADC", novel_top10, novel_top3, "novel_bucket"),
])

display(decision_validation)

bio_ids = set(bio_top10["clone_id"])
novel_ids = set(novel_top10["clone_id"])

print("=== Mode separation ===")
print("Top10 overlap:", len(bio_ids & novel_ids))
print("Biosimilar-only:", sorted(list(bio_ids - novel_ids))[:10])
print("Novel-only:", sorted(list(novel_ids - bio_ids))[:10])

,mode,top10_n,top3_n,top10_rescue_count,top3_rescue_count,top10_true_stable_rate,top3_true_stable_rate,top10_mean_true_late_qp,top10_mean_true_qp_drop,top10_mean_true_late_agg,top10_mean_pred_rescue_score
0,biosimilar,10,3,2,1,0.6,0.666667,0.000002,0.231109,8.006387,0.513504
1,novel/ADC,10,3,1,1,0.7,0.666667,0.000001,0.216336,4.036241,0.349417


=== Mode separation ===
Top10 overlap: 4
Biosimilar-only: ['CLONE_1834', 'CLONE_2358', 'CLONE_2520', 'CLONE_2759', 'CLONE_3254', 'CLONE_4616']
Novel-only: ['CLONE_0080', 'CLONE_0231', 'CLONE_3249', 'CLONE_3627', 'CLONE_4292', 'CLONE_4484']


## Step 7B — False-positive leakage and warning evaluation

This section evaluates whether aggressive false-positive clones enter the final selection.

In this project, an aggressive false-positive clone is a clone that may look attractive early but is likely to fail later.

We evaluate two things:

1. **Aggressive leakage**
   - how many true aggressive clones entered the selected Top-10 / Top-3?

2. **Warning coverage**
   - if aggressive clones entered the selected set, did the model warn us using `pred_aggr_prob`?

This metric is important because clone selection is not only about finding winners.  
It is also about avoiding expensive late-stage failures.

In [10]:
# --------------------------------------------------
# Step 7B — False-positive leakage and warning evaluation
# --------------------------------------------------

def add_aggressive_warning_flag(base_df, warning_frac=0.05):
    """
    Add an aggressive-warning flag based on the top predicted aggressive probabilities.

    warning_frac=0.05 means the highest-risk 5% of clones are flagged.
    """
    out = base_df.copy()

    if "pred_aggr_prob" not in out.columns:
        out["pred_aggr_prob"] = 0.0

    if "true_is_aggressive" not in out.columns:
        out["true_is_aggressive"] = 0

    out["true_is_aggressive"] = out["true_is_aggressive"].fillna(0).astype(int)

    thr = out["pred_aggr_prob"].quantile(1 - warning_frac)
    out["aggressive_warning"] = (out["pred_aggr_prob"] >= thr).astype(int)

    return out, thr


def summarize_aggressive_leakage(selection_df, mode_name, selection_name):
    """
    Evaluate aggressive false-positive leakage inside a selected clone set.
    """
    tmp = selection_df.copy()

    if "true_is_aggressive" not in tmp.columns:
        tmp["true_is_aggressive"] = 0

    if "aggressive_warning" not in tmp.columns:
        tmp["aggressive_warning"] = 0

    tmp["true_is_aggressive"] = tmp["true_is_aggressive"].fillna(0).astype(int)
    tmp["aggressive_warning"] = tmp["aggressive_warning"].fillna(0).astype(int)

    aggressive_n = int(tmp["true_is_aggressive"].sum())
    warning_n = int(tmp["aggressive_warning"].sum())

    warned_aggressive_n = int(
        ((tmp["true_is_aggressive"] == 1) & (tmp["aggressive_warning"] == 1)).sum()
    )

    return {
        "mode": mode_name,
        "selection": selection_name,
        "n_selected": len(tmp),
        "true_aggressive_n": aggressive_n,
        "true_aggressive_rate": aggressive_n / max(1, len(tmp)),
        "aggressive_warning_n": warning_n,
        "aggressive_warning_rate": warning_n / max(1, len(tmp)),
        "warned_aggressive_n": warned_aggressive_n,
        "warning_coverage_among_selected_aggressive": (
            warned_aggressive_n / max(1, aggressive_n)
        ),
    }


# Add warning flag to the full prediction dataframe first
df_fp, aggr_warning_threshold = add_aggressive_warning_flag(df, warning_frac=0.05)

print("Aggressive warning threshold:", aggr_warning_threshold)
print("Global aggressive warning rate:", df_fp["aggressive_warning"].mean())
print("Global true aggressive rate:", df_fp["true_is_aggressive"].mean())

# Carry warning flag into final selections by clone_id
fp_cols = ["clone_id", "true_is_aggressive", "pred_aggr_prob", "aggressive_warning"]

bio_top10_fp = bio_top10.drop(
    columns=[c for c in ["true_is_aggressive", "aggressive_warning"] if c in bio_top10.columns],
    errors="ignore"
).merge(df_fp[fp_cols], on="clone_id", how="left")

bio_top3_fp = bio_top3.drop(
    columns=[c for c in ["true_is_aggressive", "aggressive_warning"] if c in bio_top3.columns],
    errors="ignore"
).merge(df_fp[fp_cols], on="clone_id", how="left")

novel_top10_fp = novel_top10.drop(
    columns=[c for c in ["true_is_aggressive", "aggressive_warning"] if c in novel_top10.columns],
    errors="ignore"
).merge(df_fp[fp_cols], on="clone_id", how="left")

novel_top3_fp = novel_top3.drop(
    columns=[c for c in ["true_is_aggressive", "aggressive_warning"] if c in novel_top3.columns],
    errors="ignore"
).merge(df_fp[fp_cols], on="clone_id", how="left")

fp_decision_rows = [
    summarize_aggressive_leakage(bio_top10_fp, "biosimilar", "top10"),
    summarize_aggressive_leakage(bio_top3_fp, "biosimilar", "top3"),
    summarize_aggressive_leakage(novel_top10_fp, "novel/ADC", "top10"),
    summarize_aggressive_leakage(novel_top3_fp, "novel/ADC", "top3"),
]

fp_decision_table = pd.DataFrame(fp_decision_rows)
display(fp_decision_table)

print("=== Interpretation ===")
print("- true_aggressive_n: aggressive false-positive clones that entered selection")
print("- aggressive_warning_n: selected clones flagged as aggressive-risk")
print("- warning_coverage_among_selected_aggressive: among selected aggressive clones, how many were warned")
print("- Ideal production behavior: low true_aggressive_n, high warning coverage if any aggressive clone enters")

Aggressive warning threshold: 0.2659266796899224
Global aggressive warning rate: 0.05
Global true aggressive rate: 0.039


,mode,selection,n_selected,true_aggressive_n,true_aggressive_rate,aggressive_warning_n,aggressive_warning_rate,warned_aggressive_n,warning_coverage_among_selected_aggressive
0,biosimilar,top10,10,0,0.0,0,0.0,0,0.0
1,biosimilar,top3,3,0,0.0,0,0.0,0,0.0
2,novel/ADC,top10,10,0,0.0,0,0.0,0,0.0
3,novel/ADC,top3,3,0,0.0,0,0.0,0,0.0


=== Interpretation ===
- true_aggressive_n: aggressive false-positive clones that entered selection
- aggressive_warning_n: selected clones flagged as aggressive-risk
- warning_coverage_among_selected_aggressive: among selected aggressive clones, how many were warned
- Ideal production behavior: low true_aggressive_n, high warning coverage if any aggressive clone enters


## Step 8 — Decision validation block

This section validates whether the decision engine behaves reasonably before we move on to generator realism updates.

We check five things:

1. **Mode separation**
   - Are biosimilar and novel/ADC selections meaningfully different?

2. **Rescue volume**
   - Are rescue clone counts reasonable, rather than excessively large?

3. **Final selection quality**
   - Are the final Top-10 / Top-3 sets acceptably stable?

4. **Utility alignment**
   - Does predicted selection still overlap with true-utility winners?

5. **Pre-generator baseline**
   - These values serve as the baseline before applying late-aggregation realism patches.

This is the main validation step for the decision engine itself.

In [11]:
# --------------------------------------------------
# Step 8 — Decision validation block
# --------------------------------------------------

validation_rows = []

# ---------------------------
# Selection overlap / mode separation
# ---------------------------
bio_ids = set(bio_top10["clone_id"])
novel_ids = set(novel_top10["clone_id"])

top10_overlap_count = len(bio_ids & novel_ids)
top10_overlap_frac = top10_overlap_count / max(1, TOP_N)

# ---------------------------
# Rescue counts
# ---------------------------
bio_rescue_n_stage1 = len(bio_stage1_rescue) if "bio_stage1_rescue" in globals() else np.nan
bio_rescue_n_stage2 = len(bio_stage2_rescue) if "bio_stage2_rescue" in globals() else np.nan

novel_rescue_n_stage1 = len(novel_stage1_rescue) if "novel_stage1_rescue" in globals() else np.nan
novel_rescue_n_stage2 = len(novel_stage2_rescue) if "novel_stage2_rescue" in globals() else np.nan

# ---------------------------
# Final selection quality
# ---------------------------
bio_top10_stable_rate = float(bio_top10["true_stable_label"].mean())
bio_top3_stable_rate = float(bio_top3["true_stable_label"].mean()) if "bio_top3" in globals() else np.nan

novel_top10_stable_rate = float(novel_top10["true_stable_label"].mean())
novel_top3_stable_rate = float(novel_top3["true_stable_label"].mean()) if "novel_top3" in globals() else np.nan

# ---------------------------
# Utility overlap
# ---------------------------
bio_util_overlap = topk_overlap(df["true_util_bio"], df["pred_util_bio"], TOP_N)
novel_util_overlap = topk_overlap(df["true_util_novel"], df["pred_util_novel"], TOP_N)

# ---------------------------
# Validation table
# ---------------------------
validation_rows.append({
    "check": "mode_overlap_top10",
    "biosimilar": np.nan,
    "novel_adc": np.nan,
    "combined": top10_overlap_count,
    "notes": f"{top10_overlap_frac:.2f} of final top10 overlaps between modes",
})

validation_rows.append({
    "check": "stage1_rescue_n",
    "biosimilar": bio_rescue_n_stage1,
    "novel_adc": novel_rescue_n_stage1,
    "combined": np.nan,
    "notes": "Too high may indicate thresholds are too loose",
})

validation_rows.append({
    "check": "stage2_rescue_n",
    "biosimilar": bio_rescue_n_stage2,
    "novel_adc": novel_rescue_n_stage2,
    "combined": np.nan,
    "notes": "Final rescue count should remain limited",
})

validation_rows.append({
    "check": "final_top10_stable_rate",
    "biosimilar": bio_top10_stable_rate,
    "novel_adc": novel_top10_stable_rate,
    "combined": np.nan,
    "notes": "Higher is better",
})

validation_rows.append({
    "check": "final_top3_stable_rate",
    "biosimilar": bio_top3_stable_rate,
    "novel_adc": novel_top3_stable_rate,
    "combined": np.nan,
    "notes": "Higher is better; should usually be >= top10 stable rate",
})

validation_rows.append({
    "check": "top10_true_utility_overlap",
    "biosimilar": bio_util_overlap,
    "novel_adc": novel_util_overlap,
    "combined": np.nan,
    "notes": "Ranking alignment with true utility",
})

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,check,biosimilar,novel_adc,combined,notes
0,mode_overlap_top10,NaN,NaN,4.0,0.40 of final top10 overlaps between modes
1,stage1_rescue_n,225.000000,189.000000,NaN,Too high may indicate thresholds are too loose
2,stage2_rescue_n,5.000000,5.000000,NaN,Final rescue count should remain limited
3,final_top10_stable_rate,0.600000,0.700000,NaN,Higher is better
4,final_top3_stable_rate,0.666667,0.666667,NaN,Higher is better; should usually be >= top10 s...
5,top10_true_utility_overlap,0.500000,0.400000,NaN,Ranking alignment with true utility


In [12]:
print("=== Decision validation summary ===")
print(f"Final top10 overlap between biosimilar and novel/ADC: {top10_overlap_count}/{TOP_N} ({top10_overlap_frac:.2%})")

print("\nRescue counts")
print(f"- Biosimilar: Stage1={bio_rescue_n_stage1}, Stage2={bio_rescue_n_stage2}")
print(f"- Novel/ADC: Stage1={novel_rescue_n_stage1}, Stage2={novel_rescue_n_stage2}")

print("\nFinal stable rates")
print(f"- Biosimilar: top10={bio_top10_stable_rate:.3f}, top3={bio_top3_stable_rate:.3f}")
print(f"- Novel/ADC: top10={novel_top10_stable_rate:.3f}, top3={novel_top3_stable_rate:.3f}")

print("\nUtility overlap")
print(f"- Biosimilar top10 utility overlap: {bio_util_overlap:.3f}")
print(f"- Novel/ADC top10 utility overlap: {novel_util_overlap:.3f}")

=== Decision validation summary ===
Final top10 overlap between biosimilar and novel/ADC: 4/10 (40.00%)

Rescue counts
- Biosimilar: Stage1=225, Stage2=5
- Novel/ADC: Stage1=189, Stage2=5

Final stable rates
- Biosimilar: top10=0.600, top3=0.667
- Novel/ADC: top10=0.700, top3=0.667

Utility overlap
- Biosimilar top10 utility overlap: 0.500
- Novel/ADC top10 utility overlap: 0.400


## Step 9 — Utility-based weight sweep (optional analysis)

This section is not the main deployment rule.

Instead, it helps us explore whether different utility weight combinations
could improve top-k clone selection.

This is useful for:
- comparing biosimilar vs novel priorities
- tuning future SDL decision rules
- stress-testing the decision engine

In [13]:
grid_bio = {
    "w_qp": [0.5, 1.0, 1.5, 2.0],
    "w_drop": [0.5, 1.0, 1.5, 2.0],
    "w_agg": [0.0, 0.1, 0.2, 0.3],
}

grid_novel = {
    "w_qp": [0.5, 1.0, 1.5],
    "w_drop": [0.5, 1.0, 1.5],
    "w_agg": [0.5, 1.0, 1.5, 2.0],
}

ELITE_FRACS = [0.10]
K_LIST = [5, 10, 20]

def mark_elite(df, util_col, frac):
    thr = df[util_col].quantile(1 - frac)
    return (df[util_col] >= thr).astype(int)

def run_sweep(base_df, mode_name, grid, true_util_col):
    rows = []
    work = base_df.copy()

    for w_qp in grid["w_qp"]:
        for w_drop in grid["w_drop"]:
            for w_agg in grid["w_agg"]:
                work["pred_util_tmp"] = (
                    w_qp * z(work["pred_late_qp"])
                    - w_drop * z(work["pred_qp_drop"])
                    - w_agg * z(work["pred_late_agg"])
                )

                for frac in ELITE_FRACS:
                    elite_col = f"elite_{int(frac*100)}"
                    work[elite_col] = mark_elite(work, true_util_col, frac)

                    for k in K_LIST:
                        rows.append({
                            "mode": mode_name,
                            "w_qp": w_qp,
                            "w_drop": w_drop,
                            "w_agg": w_agg,
                            "elite_frac": frac,
                            "k": k,
                            "precision@k": precision_at_k_df(work, "pred_util_tmp", elite_col, k),
                            "ndcg@k": ndcg_at_k_df(work, "pred_util_tmp", elite_col, k),
                        })
    return pd.DataFrame(rows)

bio_sweep = run_sweep(df, "biosimilar", grid_bio, "true_util_bio")
novel_sweep = run_sweep(df, "novel/ADC", grid_novel, "true_util_novel")

display(bio_sweep.sort_values(["precision@k", "ndcg@k"], ascending=False).head(10))
display(novel_sweep.sort_values(["precision@k", "ndcg@k"], ascending=False).head(10))

,mode,w_qp,w_drop,w_agg,elite_frac,k,precision@k,ndcg@k
48,biosimilar,1.0,0.5,0.0,0.1,5,1.0,1.0
51,biosimilar,1.0,0.5,0.1,0.1,5,1.0,1.0
54,biosimilar,1.0,0.5,0.2,0.1,5,1.0,1.0
57,biosimilar,1.0,0.5,0.3,0.1,5,1.0,1.0
96,biosimilar,1.5,0.5,0.0,0.1,5,1.0,1.0
99,biosimilar,1.5,0.5,0.1,0.1,5,1.0,1.0
102,biosimilar,1.5,0.5,0.2,0.1,5,1.0,1.0
105,biosimilar,1.5,0.5,0.3,0.1,5,1.0,1.0
144,biosimilar,2.0,0.5,0.0,0.1,5,1.0,1.0
147,biosimilar,2.0,0.5,0.1,0.1,5,1.0,1.0


,mode,w_qp,w_drop,w_agg,elite_frac,k,precision@k,ndcg@k
0,novel/ADC,0.5,0.5,0.5,0.1,5,1.0,1.0
6,novel/ADC,0.5,0.5,1.5,0.1,5,1.0,1.0
51,novel/ADC,1.0,1.0,1.0,0.1,5,1.0,1.0
93,novel/ADC,1.5,1.0,2.0,0.1,5,1.0,1.0
102,novel/ADC,1.5,1.5,1.5,0.1,5,1.0,1.0
3,novel/ADC,0.5,0.5,1.0,0.1,5,0.8,1.0
9,novel/ADC,0.5,0.5,2.0,0.1,5,0.8,1.0
15,novel/ADC,0.5,1.0,1.0,0.1,5,0.8,1.0
36,novel/ADC,1.0,0.5,0.5,0.1,5,0.8,1.0
39,novel/ADC,1.0,0.5,1.0,0.1,5,0.8,1.0


## Output of Notebook 04

This notebook converts predicted late-stage outcomes into rescue-aware clone selection decisions.

### Main outputs

- Biosimilar-mode pass / rescue / fail buckets
- Novel / ADC-mode pass / rescue / fail buckets
- Stage 2 re-ranked candidate pools
- Final Top-10 and Top-3 selections
- Rescue-aware decision validation
- Optional utility weight sweep

### Decision logic

- **Pass**
  - predicted to satisfy productivity, stability, and quality requirements
- **Rescue**
  - not an obvious pass clone
  - has strong predicted upside
  - has process-optimization potential
- **Fail**
  - insufficient predicted performance or excessive risk

### Pipeline meaning

- **Notebook 03 = prediction engine**
- **Notebook 04 = rescue-aware decision engine**

### Final baseline decision rule

The final decision engine uses pure ranking for Top-3 production candidates, while applying a minimal rescue guardrail only to the Top-10 biosimilar candidate pool.

This preserves production-candidate integrity while preventing high-upside rescue candidates from being completely excluded.

Current baseline settings:

- Biosimilar mode:
  - Top-10 uses ranking plus a minimal rescue guardrail
  - At least one rescue candidate is retained if no rescue clone appears naturally
  - Top-3 remains pure ranking

- Novel / ADC mode:
  - Top-10 remains pure ranking
  - No forced rescue inclusion
  - Top-3 remains pure ranking

This notebook is the bridge from ML prediction to practical CLD decision-making.